In [ ]:
# model_evaluation.ipynb
# Evaluate trained models using accuracy, precision, recall, F1, and confusion matrices
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [ ]:
# Clone repo
!git clone https://github.com/KeXen1/Student-Performance-ML-Project.git

%cd Student-Performance-ML-Project

In [ ]:
# Load cleaned dataset
data = pd.read_csv("data/processed/cleaned_data.csv")

display(data.head())
print(data.shape)

In [ ]:
# Build target variable
def grade_category(g3):
    if g3 < 10:
        return "Low"
    elif g3 < 15:
        return "Medium"
    else:
        return "High"

data["performance_category"] = data["G3"].apply(grade_category)

X = data.drop(columns=["G3", "G2", "performance_category"], errors="ignore")
y = data["performance_category"]

In [ ]:
# Recreate the same train/test split used in model_training.ipynb
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=0,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

In [ ]:
# Load trained models
model_files = {
    "Logistic Regression": "models/logistic_regression.pkl",
    "Decision Tree": "models/decision_tree.pkl",
    "Random Forest": "models/random_forest.pkl"
}

trained_models = {}
for name, path in model_files.items():
    trained_models[name] = joblib.load(path)
    print("Loaded:", name)

In [ ]:
# Evaluate each model on the test set
evaluation_results = []
predictions = {}
confusion_matrices = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    cm = confusion_matrix(y_test, y_pred, labels=["Low", "Medium", "High"])
    confusion_matrices[name] = cm

    evaluation_results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

    print("=" * 60)
    print(name)
    print("-" * 60)
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)
    print()
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Confusion Matrix (rows=actual, cols=predicted, order=[Low, Medium, High]):")
    print(cm)
    print()

In [ ]:
# Compare model performance
results_df = pd.DataFrame(evaluation_results).sort_values(by="F1 Score", ascending=False).reset_index(drop=True)
display(results_df)

best_model = results_df.iloc[0]["Model"]
print("Best model based on F1 score:", best_model)

In [ ]:
# Save evaluation results
os.makedirs("results", exist_ok=True)

results_df.to_csv("results/evaluation_metrics.csv", index=False)

for name, cm in confusion_matrices.items():
    cm_df = pd.DataFrame(
        cm,
        index=["Actual Low", "Actual Medium", "Actual High"],
        columns=["Predicted Low", "Predicted Medium", "Predicted High"]
    )
    safe_name = name.lower().replace(" ", "_")
    cm_df.to_csv(f"results/confusion_matrix_{safe_name}.csv")

print("Evaluation metrics saved to results/evaluation_metrics.csv")
print("Confusion matrices saved to results/confusion_matrix_<model>.csv")